<a href="https://colab.research.google.com/github/tmzt/TrainingExperiments/blob/main/Highbay/Local/HighbaySchemaProseFinetune1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive

# 1. Mount Google Drive
# Note: This will prompt for authorization.
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# Pin torchao to a stable version (> 0.16.0) compatible with PEFT
!pip install torchao==0.18.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 67.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
# 1. Install Unsloth and dependencies
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps unsloth_zoo
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes

# import torch
# from unsloth import FastLanguageModel
# from datasets import load_dataset
# from google.colab import drive
# from trl import SFTTrainer
# from transformers import TrainingArguments
# from unsloth.chat_templates import get_chat_template
# import os


  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-wkzxxdhg/unsloth_112685e4263a4804b941c78e19a0254c
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-wkzxxdhg/unsloth_112685e4263a4804b941c78e19a0254c
  Resolved https://github.com/unslothai/unsloth.git to commit 4bc460c3a76238649ac910097e1f9b04c0d49244
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.8.3-py3-none-any.whl size=41251040 sha256=83a6bbca704bf7fa44b08dfe215bb5772db9c5e77ffc9cbf9100acfe8a227a8c
  Stored in directory: /tmp/pip-ephem-wheel-cache-nf5kwpny/wheels/60/3e/1f/e576c07051d90cf64b6a41434d87ccf4db33fafd5343bf5de0
Successfully built unsloth
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 97.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 22.9 MB/s eta 0:00:00
   ━━━━━━

In [ ]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from google.colab import drive
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth.chat_templates import get_chat_template
import os


# 3. Load 4-bit Llama-3.2-3B-Instruct
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# 4. Configure LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.3: Fast Llama patching. Transformers: 5.13.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.8.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
# 5. Load and map JSONL dataset from Drive
output_drive_path = "/content/drive/MyDrive/Training Data/"
os.makedirs(output_drive_path, exist_ok=True)

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
    mapping = {"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"},
)

def formatting_prompts_func(examples):
    all_conversations = []
    for user_input, output_ast in zip(examples["user_input"], examples["output_ast"]):
        conversation = []
        if user_input:
            conversation.append({"role": "user", "content": user_input})

        ui_prompt = None
        if output_ast and isinstance(output_ast, dict):
            ui_prompt = output_ast.get('ui_prompt')

        if ui_prompt:
            conversation.append({"role": "assistant", "content": ui_prompt})

        all_conversations.append(conversation)

    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in all_conversations]
    return { "text" : texts, }

dataset_path = os.path.join(output_drive_path, "mobile_data_prompts.jsonl")
dataset = load_dataset("json", data_files = dataset_path, split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True)

Unsloth: Restored added_tokens_decoder metadata in /content/_unsloth_sentencepiece_temp/tokenizer_5egoabcl/tokenizer_config.json.


Map:   0%|          | 0/92 [00:00<?, ? examples/s]

In [ ]:
import json

print(f"Attempting to debug schema inconsistency in: {dataset_path}")

line_count = 0
max_lines_to_check = 10 # Check the first 10 lines

with open(dataset_path, 'r', encoding='utf-8-sig') as f:
    for i, line in enumerate(f):
        line_count += 1
        if line_count > max_lines_to_check:
            break

        try:
            parsed_json = json.loads(line)
            # Safely access the problematic field and print its type

            try:
                output_ast = parsed_json.get('output_ast')
                if output_ast is None:
                    print(f"Line {i+1}: 'output_ast' is None or missing.")
                    continue

                pipeline_ast = output_ast.get('pipeline_ast')
                if pipeline_ast is None:
                    print(f"Line {i+1}: 'pipeline_ast' is None or missing.")
                    continue

                conditions = pipeline_ast.get('conditions')
                if conditions is None:
                    print(f"Line {i+1}: 'conditions' is None or missing.")
                    continue

                if isinstance(conditions, list):
                    for j, condition_item in enumerate(conditions):
                        if isinstance(condition_item, dict) and 'value' in condition_item:
                            condition_value = condition_item['value']
                            print(f"Line {i+1}, Condition {j}: 'conditions[].value' type is {type(condition_value).__name__}")
                        else:
                            print(f"Line {i+1}, Condition {j}: 'conditions[]' item is not a dict or 'value' key is missing.")
                else:
                    print(f"Line {i+1}: 'conditions' is not a list (type: {type(conditions).__name__}).")

            except KeyError:
                print(f"Line {i+1}: A key in the 'output_ast.pipeline_ast.conditions[].value' path was not found.")
            except Exception as e:
                print(f"Line {i+1}: An unexpected error occurred while checking conditions[].value: {e}")

        except json.JSONDecodeError as e:
            print(f"JSON decoding error on line {i+1}: {e}")
            print(f"Problematic line content: {line.strip()}")
            break
    else:
        print("No JSON decoding errors found with `json.loads` on individual lines.")

print(f"\nFinished checking the first {line_count-1} lines (or until first error).")
print("Please inspect the output above to find where the 'conditions[].value' type changes.")

Attempting to debug schema inconsistency in: /content/drive/MyDrive/Training Data/mobile_data_prompts.jsonl
Line 1: 'pipeline_ast' is None or missing.
Line 2, Condition 0: 'conditions[].value' type is int
Line 3: 'pipeline_ast' is None or missing.
Line 4: 'conditions' is None or missing.
Line 5: 'pipeline_ast' is None or missing.
Line 6: 'conditions' is None or missing.
Line 7: 'pipeline_ast' is None or missing.
Line 8: 'conditions' is None or missing.
Line 9: 'pipeline_ast' is None or missing.
Line 10, Condition 0: 'conditions[].value' type is str

Finished checking the first 10 lines (or until first error).
Please inspect the output above to find where the 'conditions[].value' type changes.


### Preprocessing `mobile_data_prompts.jsonl` to normalize `payload`

This cell will convert all non-string `payload` fields within your dataset to their JSON string representation to ensure schema consistency for `load_dataset`.

In [ ]:
import json
import os

# Assuming dataset_path is already defined from a previous cell
# If not, uncomment and set it:
# output_drive_path = "/content/drive/MyDrive/Training Data/"
# dataset_path = os.path.join(output_drive_path, "mobile_data_prompts.jsonl")

normalized_data = []
print(f"Normalizing 'payload' and 'conditions[].value' types in: {dataset_path}")

try:
    with open(dataset_path, 'r', encoding='utf-8-sig') as f:
        for i, line in enumerate(f):
            try:
                parsed_json = json.loads(line)

                # Safely get to the payload field and normalize it
                output_ast = parsed_json.get('output_ast')
                if output_ast and output_ast.get('pipeline_ast'):
                    pipeline_ast = output_ast.get('pipeline_ast')
                    if pipeline_ast:
                        # Normalize 'payload'
                        action = pipeline_ast.get('action')
                        if action and 'payload' in action:
                            current_payload = action['payload']
                            if not isinstance(current_payload, str):
                                # Convert non-string payload to JSON string
                                action['payload'] = json.dumps(current_payload)

                        # Normalize 'conditions[].value'
                        conditions = pipeline_ast.get('conditions')
                        if isinstance(conditions, list):
                            for j, condition_item in enumerate(conditions):
                                if isinstance(condition_item, dict) and 'value' in condition_item:
                                    current_value = condition_item['value']
                                    if not isinstance(current_value, str):
                                        # Convert non-string value to JSON string
                                        condition_item['value'] = json.dumps(current_value)

                normalized_data.append(json.dumps(parsed_json))

            except json.JSONDecodeError as e:
                print(f"Skipping line {i+1} due to JSON decoding error: {e}")
                print(f"Problematic line content: {line.strip()}")
            except Exception as e:
                print(f"Skipping line {i+1} due to unexpected error during normalization: {e}")

    # Overwrite the original file with normalized data
    with open(dataset_path, 'w', encoding='utf-8') as f:
        for item in normalized_data:
            f.write(item + '\n')
    print(f"Successfully normalized and updated {dataset_path}")
    print("Please re-run the dataset loading cell (ID: `0f4ae5b2`) now.")

except FileNotFoundError:
    print(f"Error: Dataset file not found at {dataset_path}")
except Exception as e:
    print(f"An error occurred during file processing: {e}")

Normalizing 'payload' and 'conditions[].value' types in: /content/drive/MyDrive/Training Data/mobile_data_prompts.jsonl
Successfully normalized and updated /content/drive/MyDrive/Training Data/mobile_data_prompts.jsonl
Please re-run the dataset loading cell (ID: `0f4ae5b2`) now.


The output above should pinpoint the exact line where the JSON structure breaks the consistency. Once you identify the problematic line (or lines), you'll need to manually edit `mobile_data_prompts.jsonl` in your Google Drive to ensure that all entries conform to a consistent JSON schema.

After correcting the file, please rerun the cell that loads the dataset (the one with ID `0f4ae5b2`).

In [ ]:
# 6. Set up and Run Trainer
trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'

In [ ]:
# 7. Save directly to Drive
print("Saving model and GGUF directly to Drive...")
model.save_pretrained(os.path.join(output_drive_path, "lora_model"))
model.save_pretrained_gguf(os.path.join(output_drive_path, "model_q4_k_m"), tokenizer, quantization_method = "q4_k_m")
print("Fine-tuning and export complete.")

In [ ]:
import unsloth_zoo
import importlib.metadata

try:
    version = importlib.metadata.version('unsloth_zoo')
    print(f'✅ unsloth_zoo is correctly installed and importable!')
    print(f'Version: {version}')
except Exception as e:
    print(f'❌ Verification failed: {e}')